# Preprocessing Methods Experimented

In this notebook, we are experimenting with different text preprocessing techniques to determine which pipeline works best for training our emotion classification model. The following methods are being tested:

- Basic Cleaning Only
  
  Lowercasing, removing punctuation, and numbers.
- Basic Cleaning + Stopword Removal
  
  Apply basic cleaning, then remove common stopwords to reduce noise.
- Basic Cleaning + Lemmatization
  
  Apply basic cleaning and lemmatize words to their root form.
- Basic Cleaning + Stemming
  
  Apply basic cleaning and stem words to their base form.
- Basic Cleaning + Stopword Removal + Lemmatization
  
  Combine stopword removal with lemmatization after basic cleaning.
- Basic Cleaning + Stopword Removal + Stemming
  
  Combine stopword removal with stemming after basic cleaning.

Goal: Compare these preprocessing pipelines using a simple model to see which approach maximizes performance and preserves meaningful text features.##

In [1]:
import pandas as pd
import numpy as np
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.feature_selection import VarianceThreshold
import contractions

# Load Data

In [2]:
df_1 = pd.read_csv('../data/raw/goemotions_1.csv') 
df_2 = pd.read_csv('../data/raw/goemotions_2.csv')
df_3 = pd.read_csv('../data/raw/goemotions_3.csv')

df = pd.concat([df_1, df_2, df_3], ignore_index=True, axis=0)

# Basic Cleaning

In here we are first lowercasing the texts because we want to normalize to all the texts to same Because there might be upper case and lower case characters here. And we have to remove punctuations & numbers to make this more cleaner. And also in here we have removed auxilury words since they aren't very necessary.

In [3]:
def basic_clean(text):
  text = text.lower()
  text = re.sub(r'[^a-z\s]', '', text)
  return text

df['clean_text'] = df['text'].apply(basic_clean)
#df['clean_text'] = df['clean_text'].apply(lambda x: contractions.fix(x))
df.head()

# Remove auxiliary verbs
aux_verbs = set(['be', 'am', 'is', 'are', 'was', 'were', 'being', 'been',
                 'have', 'has', 'had', 'having',
                 'do', 'does', 'did', 'doing',
                 'will', 'would', 'shall', 'should', 'may', 'might', 'must', 'can', 'could'])
def remove_aux_verbs(text):
    tokens = text.split()
    filtered_tokens = [word for word in tokens if word not in aux_verbs]
    return ' '.join(filtered_tokens)
df['clean_text'] = df['clean_text'].apply(remove_aux_verbs)

# Stopword Removal
Since we want to compare texts with and without stopword we have to first remove stopwords.
Stopwords are common high frequency words like "the", "is", "and", "in". By removing these model can focus on crucial words.

In [4]:
stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
  return " ".join([word for word in text.split() if word not in stop_words])

df['no_stopwords'] = df['clean_text'].apply(remove_stopwords)

# Stemming vs Lemmatization

| Original |	Stemmed	| Lemmatized |
|----------|----------|------------|
| running	| run	| run |
| better	| better| good |

In here we are planning to compare model results from stemming vs lemmatization.

In [5]:
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def stem_text(text):
  return " ".join([stemmer.stem(word) for word in text.split()])

def lemmatize_text(text):
  return " ".join([lemmatizer.lemmatize(word) for word in text.split()])

df['stemmed'] = df['clean_text'].apply(stem_text)
df['lemmatized'] = df['clean_text'].apply(lemmatize_text)
df['stop_stemmed'] = df['no_stopwords'].apply(stem_text)
df['stop_lemmatized'] = df['no_stopwords'].apply(lemmatize_text)

# Target Processing

In [6]:
df.columns

Index(['text', 'id', 'author', 'subreddit', 'link_id', 'parent_id',
       'created_utc', 'rater_id', 'example_very_unclear', 'admiration',
       'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion',
       'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust',
       'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy',
       'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief',
       'remorse', 'sadness', 'surprise', 'neutral', 'clean_text',
       'no_stopwords', 'stemmed', 'lemmatized', 'stop_stemmed',
       'stop_lemmatized'],
      dtype='str')

In [7]:
emotions = ['admiration',
       'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion',
       'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust',
       'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy',
       'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief',
       'remorse', 'sadness', 'surprise']
# Let's turn the these labels into a 3 labels dataset: positive, negative, neutral
sentiment_dict = pd.read_json('../data/sentiment_dict.json', typ='series').to_dict()
# In this sentiment dictionary, we have the following mapping:
#{
#"positive": ["amusement", "excitement", "joy", "love", "desire", "optimism", "caring", "pride", "admiration", "gratitude", "relief", "approval"],
#"negative": ["fear", "nervousness", "remorse", "embarrassment", "disappointment", "sadness", "grief", "disgust", "anger", "annoyance", "disapproval"],
#"ambiguous": ["realization", "surprise", "curiosity", "confusion"]
#}
def label_emotion(row):
    active_emotions = [emotion for emotion in emotions if row.get(emotion, 0) == 1]

    if not active_emotions:
        return 'neutral'

    sentiment_counts = {
        'positive': 0,
        'negative': 0,
        'ambiguous': 0
    }

    for emotion in active_emotions:
        for sentiment, emotion_list in sentiment_dict.items():
            if emotion in emotion_list:
                sentiment_counts[sentiment] += 1

    # Decide based on majority
    max_sentiment = max(sentiment_counts, key=sentiment_counts.get)

    # Optional: handle ties
    # 
    if list(sentiment_counts.values()).count(sentiment_counts[max_sentiment]) > 1:
       return 'mixed'

    return max_sentiment

df['sentiment'] = df.apply(label_emotion, axis=1)
# let's remove records with 'neutral' sentiment to focus on clear positive and negative examples
df = df[df['sentiment'] != 'neutral']
df = df[df['sentiment'] != 'mixed']

sentiment_mapping = {
       'ambiguous':0,
       'positive':1,
       'negative':2
}
df['sentiment_label'] = df['sentiment'].map(sentiment_mapping)
df.head()

target = df['sentiment_label']


# Experiment with Multiple Preprocessing Pipelines

In [8]:
pipelines = {
  'Basic': df['clean_text'],
  'No Stopwords': df['no_stopwords'],
  'Stemmed': df['stemmed'],
  'Lemmatized': df['lemmatized'],
  'Stopwords Removed + Stemmed': df['stop_stemmed'],
  'Stopwords Removed + Lemmatized': df['stop_lemmatized']
}

In [9]:
results = []

# 'target' is multiclass (0=neutral, 1=ambiguous, 2=positive, 3=negative)
for name, text in pipelines.items():
    print(f'Processing pipeline: {name}')
    
    vectorizer = TfidfVectorizer(max_features=5000, min_df=5, max_df=0.8)
    X_vect = vectorizer.fit_transform(text)

    vrthr = VarianceThreshold(threshold=0.0001)
    X_vect = vrthr.fit_transform(X_vect)
    
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_vect, target.values, test_size=0.1, random_state=42, stratify=target.values
    )
    
    model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    model.fit(X_tr, y_tr)
    
    y_pred = model.predict(X_te)
    
    acc = accuracy_score(y_te, y_pred)
    precision = precision_score(y_te, y_pred, average='macro')
    recall = recall_score(y_te, y_pred, average='macro')
    f1 = f1_score(y_te, y_pred, average='macro')
    
    results.append({
        'Pipeline': name,
        'Accuracy': acc,
        'Precision': precision,
        'Recall': recall,
        'F1 Macro': f1,
    })
    
    print(f"{name} - Accuracy: {acc:.4f}, F1 Macro: {f1:.4f}")

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by='F1 Macro', ascending=False)
results_df

Processing pipeline: Basic
Basic - Accuracy: 0.7552, F1 Macro: 0.7005
Processing pipeline: No Stopwords
No Stopwords - Accuracy: 0.7497, F1 Macro: 0.6916
Processing pipeline: Stemmed
Stemmed - Accuracy: 0.7564, F1 Macro: 0.7006
Processing pipeline: Lemmatized
Lemmatized - Accuracy: 0.7550, F1 Macro: 0.6991
Processing pipeline: Stopwords Removed + Stemmed
Stopwords Removed + Stemmed - Accuracy: 0.7506, F1 Macro: 0.6928
Processing pipeline: Stopwords Removed + Lemmatized
Stopwords Removed + Lemmatized - Accuracy: 0.7507, F1 Macro: 0.6939


,Pipeline,Accuracy,Precision,Recall,F1 Macro
2,Stemmed,0.756382,0.713449,0.692192,0.700569
0,Basic,0.755244,0.713163,0.692126,0.700469
3,Lemmatized,0.754960,0.712369,0.690557,0.699130
5,Stopwords Removed + Lemmatized,0.750693,0.706560,0.685530,0.693876
4,Stopwords Removed + Stemmed,0.750622,0.705471,0.684643,0.692786
1,No Stopwords,0.749698,0.704113,0.683594,0.691639


### **Preprocessing Pipeline Comparison Results**

The performance of different preprocessing pipelines was evaluated using multiple metrics, including **Accuracy, Precision, Recall, F1**.

| Pipeline | Method | Accuracy | Precision | Recall | F1 | 
|----------|--------|----------|------------|--------------|--------------|
| 1 | Stemmed | **0.7564** | **0.7134**| **0.6922** | **0.7006** |
| 5 | Basic| 0.7552 | 0.7132 | 0.6921 | 0.7005 |
| 4 | Lemmatized | 0.7550 | 0.7124 | 0.6906 | 0.6991 |
| 0 | Stopwords Removed + Lemmatized | 0.7507 | 0.7066 | 0.6855 | 0.6928 |
| 3 | Stopwords Removed + Stemmed | 0.7506 | 0.7055 | 0.6846 | 0.6928 |
| 2 | No Stopwords | 0.7497 | 0.7041 | 0.6836 | 0.6916 |

---

### **Key Observations**

- The **“Stemmed”** pipeline achieved the **highest Accuracy, Precision, Recall and F1**, indicating better overall multi-label classification performance.
- **No Stopwords consistently resulted in lower performance**, suggesting that it removes important contextual information.

---

### **Conclusion**

Based on these metrics, the best-performing preprocessing method is:

👉 **Basic Cleaning + Stopword Removal**

---

### **Final Selected Pipeline**

> **Lowercasing + Removing punctuation + Removing numbers + Stemmed**

This preprocessing pipeline provides the best balance between **noise reduction** and **preservation of meaningful information**, leading to improved model performance.